In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

In [2]:
case1_dir = '/glade/derecho/scratch/brendanc/fates_output/radiation/FATES_case1'
case2_dir = '/glade/derecho/scratch/brendanc/fates_output/radiation/FATES_case2'
ssp_dir = '/glade/derecho/scratch/brendanc/fates_output/radiation/FATES_ssp245'
temp_dir = '/glade/derecho/scratch/brendanc/fates_output/radiation/FATES_temperature'
precip_dir = '/glade/derecho/scratch/brendanc/fates_output/radiation/FATES_precip'

case1 = xr.open_dataset(case1_dir + "/fates_G6_rad_case1_ctsm5.3.085.2035-2069.clm2.h1a.NPP.nc")
case1 = case1["FATES_NPP_latlon"]
case2 = xr.open_dataset(case2_dir + "/fates_G6_rad_case2_ctsm5.3.085.2035-2069.clm2.h1a.NPP.nc")
case2 = case2["FATES_NPP_latlon"] 
ssp = xr.open_dataset(ssp_dir + "/fates_SSP245_ctsm5.3.085_tranCO2_2015-2069.clm2.h1a.NPP.nc")
ssp = ssp["FATES_NPP_latlon"] 
temp = xr.open_dataset(temp_dir + "/fates_G6_temperature_ctsm5.3.085.2035-2069.clm2.h1a.NPP_latlon.nc")
temp = temp["FATES_NPP_latlon"]
precip = xr.open_dataset(precip_dir + "/fates_G6_precip_ctsm5.3.085.2035-2069.clm2.h1a.NPP_latlon.nc")
precip = precip["FATES_NPP_latlon"]

In [ ]:
# Time slice first
case1 = case1.sel(time=slice('2050-01-01', '2069-12-31'))
case2 = case2.sel(time=slice('2050-01-01', '2069-12-31'))
ssp = ssp.sel(time=slice('2050-01-01', '2069-12-31'))
temp = temp.sel(time=slice('2050-01-01', '2069-12-31'))
precip = precip.sel(time=slice('2050-01-01', '2069-12-31'))

case1 = case1.mean(dim=("time","fates_levscls"), skipna=True)
case2 = case2.mean(dim=("time","fates_levscls"), skipna=True)
ssp = ssp.mean(dim=("time","fates_levscls"), skipna=True)
temp = temp.mean(dim=("time","fates_levscls"), skipna=True)
precip = precip.mean(dim=("time","fates_levscls"), skipna=True)

# convert: kg m-2 s-1 -> g C m-2 yr-1
conv = 10000 * 315360000  

case1 = case1 * conv
case2 = case2 * conv
ssp = ssp * conv
temp = temp * conv
precip = precip * conv

In [ ]:
tot_rad_dif = (case1 - ssp) #/ (ssp)
#tot_rad_dif = tot_rad_dif * 100
dirdif_dif = (case2 - case1) #/ (case1)
#dirdif_dif = dirdif_dif * 100
temp_dif = (temp - ssp) #/ (ssp)
#temp_dif = temp_dif * 100
precip_dif = (precip - ssp) #/ (ssp)
#precip_dif = precip_dif * 100

In [ ]:
filedir2 = '/glade/derecho/scratch/brendanc/fates_output/radiation'
surf  = xr.open_dataset(filedir2 + "/surfdata_0.9x1.25_hist_2000_16pfts_c240908.nc")
surf = surf['PCT_NAT_PFT'].sel(natpft=0)
surf = surf.rename({
    "lsmlat": "lat",
    "lsmlon": "lon"
})
surf_binary = xr.where(surf < 30, 1, 0).astype("int8")
tot_rad_dif = tot_rad_dif.where(surf_binary == 1)
dirdif_dif = dirdif_dif.where(surf_binary == 1)
temp_dif = temp_dif.where(surf_binary == 1)
precip_dif = precip_dif.where(surf_binary == 1)

In [ ]:
filedir = '/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/surfdata_esmf/ctsm5.3.0'
landuse = xr.open_dataset(filedir + "/surfdata_0.9x1.25_hist_2000_16pfts_c240908.nc")

In [ ]:
def median_smooth_da(da, size=3):
    arr = da.values.copy()
    nan_mask = np.isnan(arr)
    arr_filled = np.where(nan_mask, np.nanmedian(arr), arr)
    smoothed = median_filter(arr_filled, size=size)
    smoothed[nan_mask] = np.nan
    return da.copy(data=smoothed)

def compute_zonal_diffs(pft_list, size=3):
    tot_rad_pft = median_smooth_da(tot_rad_dif.sel(fates_levpft=pft_list).mean('fates_levpft'), size)
    dirdif_pft  = median_smooth_da(dirdif_dif.sel(fates_levpft=pft_list).mean('fates_levpft'),  size)
    temp_pft    = median_smooth_da(temp_dif.sel(fates_levpft=pft_list).mean('fates_levpft'),    size)
    precip_pft  = median_smooth_da(precip_dif.sel(fates_levpft=pft_list).mean('fates_levpft'),  size)
    return [tot_rad_pft.mean('lon', skipna=True),
            dirdif_pft.mean('lon', skipna=True),
            temp_pft.mean('lon', skipna=True),
            precip_pft.mean('lon', skipna=True)]

pct = landuse['PCT_NAT_PFT']
def compute_zonal_pct(clm_pfts):
    return pct.sel(natpft=clm_pfts).sum('natpft').mean('lsmlon')

if 'lat' in landuse.coords:
    pct_lat = landuse['lat'].values
elif 'LATIXY' in landuse:
    pct_lat = landuse['LATIXY'].isel(lsmlon=0).values
else:
    pct_lat = landuse['lsmlat'].values

fates_groups = [[1, 4, 5, 6], [2, 3], [13], [14]]
clm_groups   = [[4, 5, 6, 7, 8], [1, 2, 3], [13], [14]]
group_titles = ["FATES Broadleaf Trees", "FATES Needleleaf Trees",
                "FATES C3 Grass", "FATES C4 Grass"]
labels = ['Total Radiation', 'Direct/Diffuse', 'Temperature', 'Precipitation']
colors = ['tab:blue', 'tab:orange', 'tab:red', 'tab:green']

fig = plt.figure(figsize=(4.5, 16))
outer = GridSpec(4, 1, figure=fig, hspace=0.2)

n_groups = len(fates_groups)
first_ax = None 
for g, (fates_pfts, clm_pfts, group_title) in enumerate(zip(fates_groups, clm_groups, group_titles)):
    row, col = g, 0

    inner = GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[row, col],
        width_ratios=[2, 1], wspace=0.0)

    if first_ax is None:
        npp_ax = fig.add_subplot(inner[0, 0]); first_ax = npp_ax
    else:
        npp_ax = fig.add_subplot(inner[0, 0], sharey=first_ax)
    pct_ax = fig.add_subplot(inner[0, 1], sharey=first_ax)

    npp_data = compute_zonal_diffs(fates_pfts, size=3)
    npp_data = [da.where(da['lat'] >= -60) for da in npp_data]
    for da, label, color in zip(npp_data, labels, colors):
        npp_ax.plot(da, da['lat'], label=label, color=color, lw=1.5)
    npp_ax.axvline(0, color="gray", lw=0.8, ls="--")
    npp_ax.set_title(f"{group_title} ΔNPP", fontsize=10)

    pct_prof = compute_zonal_pct(clm_pfts)
    pct_ax.plot(pct_prof.values, pct_lat, color='tab:green', lw=1.8)
    pct_ax.fill_betweenx(pct_lat, 0, pct_prof.values, alpha=0.2, color='tab:green')
    pct_ax.set_title("PFT cover", fontsize=10)

    for ax in (npp_ax, pct_ax):
        ax.set_ylim(-70, 90)
        ax.grid(alpha=0.3)

    plt.setp(pct_ax.get_yticklabels(), visible=False)

    npp_ax.set_ylabel("Latitude")

    # x-axis labels only on the bottom row
    if row == n_groups - 1:
        npp_ax.set_xlabel("NPP Difference (gC/m$^2$)")
        pct_ax.set_xlabel("Cover (% of nat. veg)")

    if g == 0:
        npp_ax.legend(fontsize=8, loc='upper right')

#fig.suptitle("Zonal-mean ΔNPP (FATES) and PFT cover", fontsize=10)
fig.subplots_adjust(left=0.12, right=0.96, top=0.94, bottom=0.05)
plt.show()

#savedir = "/glade/derecho/scratch/brendanc/fates_output/radiation"
#plt.savefig(f"{savedir}/FATES_NPP_PCT_lat_lines.png", dpi=600, bbox_inches="tight")